# Phase D — TON_IoT cross-dataset transfer
### *Trustworthiness Under Compression* — does the rare-class trust collapse generalise?

Runs the **entire** CICIoT2023 pipeline on a second IoT dataset (TON_IoT network). If compression collapses calibration AND faithfulness on TON_IoT's rare/hard classes too, the finding becomes a **property of compressed IoT IDS**, not a CICIoT2023 artefact — the credibility multiplier for top-3.

TON_IoT network: 461,043 flows, 9 attack types + normal. **MITM (1,043) is the natural rare class.** Feature hygiene: drop IP/port/identity columns (leakage guard).

Pipeline (reuses debugged Phase A–C code): download → preprocess → 5-seed M0 → M1–M3 → calibration (ECE+gap+temp) → faithfulness (SHAP agreement + perturbation, null band) → cross-dataset comparison.

> Runtime: GPU + reasonable RAM. TON is small (~460k rows) so training is fast; SHAP is the slow step.

## 0. Setup + download TON_IoT

In [1]:
!pip install -q shap
import os, gc, json, time, random, glob
import numpy as np, pandas as pd, tensorflow as tf, joblib, shap
from scipy.stats import spearmanr
from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

def seed_everything(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
seed_everything(0)
ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
if not os.path.isdir(ROOT):
    from google.colab import drive; drive.mount('/content/drive')
DROOT=f'{ROOT}/ton_iot'
P={k:f'{DROOT}/{k}' for k in ['data','models','results','logs','splits']}
P['shap']=f'{DROOT}/results/shap'
for v in P.values(): os.makedirs(v,exist_ok=True)
GLOBAL_DATA_SEED=42; SEEDS=[0,1,2,3,4]; ANCHOR_SEED=0
print('TF',tf.__version__,'| shap',shap.__version__)

Mounted at /content/drive
TF 2.20.0 | shap 0.52.0


In [2]:
# Download TON_IoT network CSV via Kaggle (upload kaggle.json once)
RAW=f"{P['data']}/train_test_network.csv"
if not os.path.exists(RAW):
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files; print('Upload kaggle.json:'); files.upload()
        os.makedirs('/root/.kaggle',exist_ok=True)
        os.system('cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json')
    os.system(f'kaggle datasets download -d alaaelmor/ton-iot-train-test-network -p "{P["data"]}" --unzip')
    # locate the csv (name may vary)
    found=glob.glob(f"{P['data']}/*.csv")
    print('downloaded csvs:',found)
    if found and not os.path.exists(RAW): RAW=found[0]
print('using', RAW)

Upload kaggle.json:


Saving kaggle.json to kaggle.json
downloaded csvs: ['/content/drive/MyDrive/Trustworthiness_IoT/ton_iot/data/Train_Test_Network.csv']
using /content/drive/MyDrive/Trustworthiness_IoT/ton_iot/data/Train_Test_Network.csv


## 1. Preprocess — feature hygiene + label map

In [3]:
df=pd.read_csv(RAW)
print('raw shape',df.shape); print('columns:',list(df.columns))
# label column is 'type' (attack category); 'label' is binary
LABELCOL='type' if 'type' in df.columns else 'Attack'
print('\nclass counts:\n', df[LABELCOL].astype(str).str.lower().value_counts())

raw shape (461043, 45)
columns: ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']

class counts:
 type
normal        300000
scanning       20000
dos            20000
injection      20000
ddos           20000
password       20000
xss            20000
ransomware     20000
backdoor       20000
mitm            1043
Name: count, dtype: int64


In [4]:
# Drop identity/leakage columns; map labels to 10 classes (Normal + 9 attacks)
DROP=[c for c in df.columns if c.lower() in
      ['src_ip','dst_ip','src_port','dst_port','ts','label','dns_query','ssl_subject',
       'ssl_issuer','http_uri','http_user_agent','http_referrer','weird_name','weird_addl',
       'http_orig_mime_types','http_resp_mime_types','uid','http_version','http_trans_depth']]
lab=df[LABELCOL].astype(str).str.lower().str.strip()
MAP={'normal':'Normal','backdoor':'Backdoor','ddos':'DDoS','dos':'DoS','injection':'Injection',
     'mitm':'MITM','password':'Password','ransomware':'Ransomware','scanning':'Scanning','xss':'XSS'}
df['y']=lab.map(MAP)
unmapped=sorted(set(lab[df['y'].isna()]))
assert not unmapped, f"unmapped labels: {unmapped}"
CLASSES=['Normal']+[c for c in sorted(df['y'].unique()) if c!='Normal']
class_to_idx={c:i for i,c in enumerate(CLASSES)}; N_CLASSES=len(CLASSES); BENIGN_IDX=0
print('classes',CLASSES)

Xdf=df.drop(columns=[c for c in DROP if c in df.columns]+[LABELCOL,'y'],errors='ignore').copy()
# TON_IoT uses '-' for missing in many string/numeric cols -> coerce
# split columns: numeric-coercible vs genuinely categorical (low cardinality)
obj=[c for c in Xdf.columns if Xdf[c].dtype=='object']
num_like=[]; cat=[]
for c in obj:
    coerced=pd.to_numeric(Xdf[c].replace('-',np.nan),errors='coerce')
    if coerced.notna().mean()>0.95:           # mostly numeric -> treat numeric
        Xdf[c]=coerced; num_like.append(c)
    elif Xdf[c].nunique()<=30:                # low-cardinality categorical -> one-hot
        cat.append(c)
    else:                                     # high-cardinality string -> drop (leakage/explosion guard)
        Xdf=Xdf.drop(columns=[c])
print('coerced-numeric:',num_like,'| one-hot:',cat)
Xdf=pd.get_dummies(Xdf, columns=cat, dummy_na=False)
Xdf=Xdf.apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
y=df['y'].map(class_to_idx).astype('int64').values
FEATURES=list(Xdf.columns); N_FEATURES=len(FEATURES)
print('X',Xdf.shape,'| features',N_FEATURES)

classes ['Normal', 'Backdoor', 'DDoS', 'DoS', 'Injection', 'MITM', 'Password', 'Ransomware', 'Scanning', 'XSS']
coerced-numeric: [] | one-hot: ['proto', 'service', 'conn_state', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'http_method', 'weird_notice']
X (461043, 74) | features 74


## 2. Locked stratified split + train-only scaler + tempered weights

In [5]:
idx=np.arange(len(y))
tr,tmp=train_test_split(idx,test_size=0.30,stratify=y,random_state=GLOBAL_DATA_SEED)
va,te=train_test_split(tmp,test_size=0.50,stratify=y[tmp],random_state=GLOBAL_DATA_SEED)
np.savez(f"{P['splits']}/split.npz",train=tr,val=va,test=te)
scaler=StandardScaler().fit(Xdf.iloc[tr]); joblib.dump(scaler,f"{P['models']}/scaler.joblib")
Xtr=scaler.transform(Xdf.iloc[tr]).astype('float32'); ytr=y[tr]
Xva=scaler.transform(Xdf.iloc[va]).astype('float32'); yva=y[va]
Xte=scaler.transform(Xdf.iloc[te]).astype('float32'); yte=y[te]
bal=compute_class_weight('balanced',classes=np.arange(N_CLASSES),y=ytr)
tw=np.sqrt(bal); tw=tw/tw.mean(); class_weight={i:float(w) for i,w in enumerate(tw)}
print('split tr/va/te',len(tr),len(va),len(te))
print('per-class test counts:',[int((yte==i).sum()) for i in range(N_CLASSES)])

split tr/va/te 322730 69156 69157
per-class test counts: [45000, 3000, 3000, 3000, 3000, 157, 3000, 3000, 3000, 3000]


## 3. M0 ×5 seeds (anchor seed 0)

In [6]:
def build_mlp(nf,nc,seed):
    seed_everything(seed); init=tf.keras.initializers.GlorotUniform(seed=seed)
    m=tf.keras.Sequential([tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(128,activation='relu',kernel_initializer=init),
        tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3,seed=seed),
        tf.keras.layers.Dense(64,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dropout(0.2,seed=seed),
        tf.keras.layers.Dense(nc,activation='softmax',kernel_initializer=init)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
for s in SEEDS:
    p=f"{P['models']}/M0_seed{s}.keras"
    if os.path.exists(p): print('exists',p); continue
    seed_everything(s); m=build_mlp(N_FEATURES,N_CLASSES,s)
    m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=60,batch_size=2048,class_weight=class_weight,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True),
                     tf.keras.callbacks.ReduceLROnPlateau(patience=3,factor=0.5)],verbose=2)
    m.save(p); print('saved',p)
json.dump({'anchor':ANCHOR_SEED,'seeds':SEEDS,'weighting':'tempered sqrt','split_seed':GLOBAL_DATA_SEED,
           'dataset':'TON_IoT network','rare_class':'MITM'},open(f"{P['logs']}/anchor.json",'w'),indent=2)
anchor_path=f"{P['models']}/M0_seed{ANCHOR_SEED}.keras"; anchor=tf.keras.models.load_model(anchor_path)
pred=anchor.predict(Xva,batch_size=8192,verbose=0).argmax(1)
print('anchor macro-F1',round(f1_score(yva,pred,average='macro'),4))
print(classification_report(yva,pred,target_names=CLASSES,digits=3))

Epoch 1/60
158/158 - 8s - 52ms/step - accuracy: 0.6239 - loss: 0.4912 - val_accuracy: 0.7427 - val_loss: 0.7160 - learning_rate: 0.0010
Epoch 2/60
158/158 - 1s - 4ms/step - accuracy: 0.7071 - loss: 0.3702 - val_accuracy: 0.7433 - val_loss: 0.6434 - learning_rate: 0.0010
Epoch 3/60
158/158 - 1s - 3ms/step - accuracy: 0.7211 - loss: 0.3550 - val_accuracy: 0.7431 - val_loss: 0.6323 - learning_rate: 0.0010
Epoch 4/60
158/158 - 1s - 3ms/step - accuracy: 0.7311 - loss: 0.3481 - val_accuracy: 0.7436 - val_loss: 0.6343 - learning_rate: 0.0010
Epoch 5/60
158/158 - 1s - 4ms/step - accuracy: 0.7385 - loss: 0.3435 - val_accuracy: 0.7441 - val_loss: 0.6310 - learning_rate: 0.0010
Epoch 6/60
158/158 - 1s - 4ms/step - accuracy: 0.7422 - loss: 0.3395 - val_accuracy: 0.7457 - val_loss: 0.6286 - learning_rate: 0.0010
Epoch 7/60
158/158 - 1s - 4ms/step - accuracy: 0.7444 - loss: 0.3369 - val_accuracy: 0.7457 - val_loss: 0.6274 - learning_rate: 0.0010
Epoch 8/60
158/158 - 1s - 4ms/step - accuracy: 0.7451 

## 4. M1–M3 (pruning, distillation, int8/fp16) — same recipes as Phase B

In [7]:
# M2 pruning
def mmasks(model,sp):
    M={}
    for i,l in enumerate(model.layers):
        if isinstance(l,tf.keras.layers.Dense):
            W=l.get_weights()[0]; M[i]=(np.abs(W)>=np.quantile(np.abs(W),sp)).astype('float32')
    return M
def amask(model,M):
    for i,m in M.items():
        w=model.layers[i].get_weights(); w[0]=w[0]*m; model.layers[i].set_weights(w)
def prune(sp,epochs=2):
    seed_everything(0); model=tf.keras.models.load_model(anchor_path); M=mmasks(model,sp); amask(model,M)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    class RM(tf.keras.callbacks.Callback):
        def on_train_batch_end(self,b,logs=None): amask(self.model,M)
    model.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=epochs,batch_size=2048,callbacks=[RM()],verbose=2)
    amask(model,M); out=f"{P['models']}/M2_prune{int(sp*100)}.keras"; model.save(out); return out
M2={s:prune(s) for s in (0.50,0.80)}

# M3 distillation (logit student, correct KD)
def student_logits(nf,nc,seed=0):
    seed_everything(seed); init=tf.keras.initializers.GlorotUniform(seed=seed)
    return tf.keras.Sequential([tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(48,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dense(24,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dense(nc,kernel_initializer=init)])
tl=np.log(np.clip(anchor.predict(Xtr,batch_size=8192,verbose=0),1e-9,1)).astype('float32')
class Distiller(tf.keras.Model):
    def __init__(s,st,T=4.,a=.5): super().__init__(); s.st=st; s.T=T; s.a=a
    def compile(s,opt): super().compile(optimizer=opt); s.ce=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True); s.kl=tf.keras.losses.KLDivergence(); s.lm=tf.keras.metrics.Mean('loss'); s.am=tf.keras.metrics.SparseCategoricalAccuracy('acc')
    def train_step(s,d):
        x,yt,t=d; ts=tf.nn.softmax(t/s.T)
        with tf.GradientTape() as tp:
            sl=s.st(x,training=True); ss=tf.nn.softmax(sl/s.T)
            loss=s.a*s.ce(yt,sl)+(1-s.a)*(s.T**2)*s.kl(ts,ss)
        g=tp.gradient(loss,s.st.trainable_variables); s.optimizer.apply_gradients(zip(g,s.st.trainable_variables))
        s.lm.update_state(loss); s.am.update_state(yt,sl); return {'loss':s.lm.result(),'acc':s.am.result()}
seed_everything(0); st=student_logits(N_FEATURES,N_CLASSES)
dd=tf.data.Dataset.from_tensor_slices((Xtr,ytr,tl)).shuffle(100000,seed=0).batch(2048).prefetch(2)
D=Distiller(st); D.compile(tf.keras.optimizers.Adam(1e-3)); D.fit(dd,epochs=15,verbose=2)
M3=tf.keras.Sequential([st,tf.keras.layers.Softmax()]); M3.build((None,N_FEATURES))
M3_path=f"{P['models']}/M3_distilled.keras"; M3.save(M3_path)

# M1 dynamic int8 + fp16
def tflite(mode):
    c=tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(anchor_path)); c.optimizations=[tf.lite.Optimize.DEFAULT]
    if mode=='fp16': c.target_spec.supported_types=[tf.float16]
    o=f"{P['models']}/M1_{mode}.tflite"; open(o,'wb').write(c.convert()); return o
M1_int8=tflite('int8'); M1_fp16=tflite('fp16')
print('variants built')

Epoch 1/2
158/158 - 7s - 43ms/step - accuracy: 0.7712 - loss: 0.6361 - val_accuracy: 0.8265 - val_loss: 0.4593
Epoch 2/2
158/158 - 1s - 8ms/step - accuracy: 0.8005 - loss: 0.5074 - val_accuracy: 0.8301 - val_loss: 0.4683
Epoch 1/2
158/158 - 8s - 50ms/step - accuracy: 0.6209 - loss: 1.0889 - val_accuracy: 0.7393 - val_loss: 0.7294
Epoch 2/2
158/158 - 1s - 9ms/step - accuracy: 0.7184 - loss: 0.7350 - val_accuracy: 0.7813 - val_loss: 0.6202
Epoch 1/15
158/158 - 5s - 31ms/step - acc: 0.6293 - loss: 4.0762
Epoch 2/15
158/158 - 2s - 10ms/step - acc: 0.7419 - loss: 0.8352
Epoch 3/15
158/158 - 2s - 11ms/step - acc: 0.7643 - loss: 0.6241
Epoch 4/15
158/158 - 2s - 11ms/step - acc: 0.7686 - loss: 0.5856
Epoch 5/15
158/158 - 2s - 11ms/step - acc: 0.7697 - loss: 0.5679
Epoch 6/15
158/158 - 2s - 11ms/step - acc: 0.7702 - loss: 0.5580
Epoch 7/15
158/158 - 2s - 11ms/step - acc: 0.7700 - loss: 0.5516
Epoch 8/15
158/158 - 2s - 11ms/step - acc: 0.7673 - loss: 0.5462
Epoch 9/15
158/158 - 2s - 11ms/step - 

## 5. Calibration — per-class ECE + signed gap + temp scaling

In [8]:
def proba(path,kind,X,batch=8192):
    if kind=='keras': return tf.keras.models.load_model(path).predict(X,batch_size=batch,verbose=0)
    it=tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    inp=it.get_input_details()[0]['index']; outd=it.get_output_details()[0]['index']
    Pr=np.zeros((len(X),N_CLASSES),'float32')
    for i in range(len(X)):
        it.set_tensor(inp,X[i:i+1].astype(it.get_input_details()[0]['dtype'])); it.invoke(); Pr[i]=it.get_tensor(outd)[0]
    Pr=np.clip(Pr,0,None); s=Pr.sum(1,keepdims=True); s[s==0]=1; return Pr/s
def ece(conf,corr,nb=15):
    o=np.argsort(conf); conf,corr=conf[o],corr[o]; e=0
    for b in np.array_split(np.arange(len(conf)),nb):
        if len(b): e+=abs(corr[b].mean()-conf[b].mean())*len(b)/len(conf)
    return e
def gap(conf,corr): return float(conf.mean()-corr.mean())
def appT(Pp,T): z=np.log(np.clip(Pp,1e-9,1))/T; z-=z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def fitT(Pp,yv):
    return float(minimize_scalar(lambda T:-np.mean(np.log(appT(Pp,T)[np.arange(len(yv)),yv]+1e-9)),bounds=(0.5,5),method='bounded').x)
VAR=[('M0',anchor_path,'keras'),('M2_prune50',M2[0.50],'keras'),('M2_prune80',M2[0.80],'keras'),
     ('M3_distilled',M3_path,'keras'),('M1_int8',M1_int8,'tflite'),('M1_float16',M1_fp16,'tflite')]
rows=[];pcs=[]
for n,p,k in VAR:
    Pv=proba(p,k,Xva); T=fitT(Pv,yva); Pt=proba(p,k,Xte); pr=Pt.argmax(1); cr=(pr==yte).astype(float); cf=Pt.max(1)
    rows.append({'model':n,'T':round(T,3),'ECE_raw':round(ece(cf,cr),4),'ECE_temp':round(ece(appT(Pt,T).max(1),cr),4),'gap_raw':round(gap(cf,cr),4)})
    d={'model':n}
    for ci,c in enumerate(CLASSES):
        m=(yte==ci); d[c]=round(gap(cf[m],cr[m]),4) if m.sum() else np.nan
    pcs.append(d)
cal=pd.DataFrame(rows); pcc=pd.DataFrame(pcs); pd.set_option('display.width',240)
print(cal.to_string(index=False)); print(); print(pcc.to_string(index=False))
cal.to_csv(f"{P['results']}/calibration.csv",index=False); pcc.to_csv(f"{P['results']}/calibration_perclass.csv",index=False)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


       model     T  ECE_raw  ECE_temp  gap_raw
          M0 1.043   0.0528    0.0493   0.0193
  M2_prune50 0.721   0.0874    0.0648  -0.0631
  M2_prune80 0.603   0.1301    0.0755  -0.0944
M3_distilled 1.129   0.0633    0.0841   0.0273
     M1_int8 4.309   0.2277    0.1751   0.2143
  M1_float16 1.038   0.0537    0.0494   0.0195

       model  Normal  Backdoor    DDoS    DoS  Injection    MITM  Password  Ransomware  Scanning     XSS
          M0  0.0605   -0.5087 -0.0027 0.3235    -0.0877 -0.3323    0.0137     -0.0321   -0.0941 -0.0558
  M2_prune50 -0.0553   -0.5157 -0.1091 0.3168    -0.0184  0.5418    0.0055      0.0346   -0.1972 -0.1701
  M2_prune80 -0.1246   -0.6590 -0.4165 0.2712    -0.0041  0.3073    0.3133      0.6683   -0.3714 -0.1250
M3_distilled  0.0078   -0.5964 -0.0201 0.2398    -0.1136  0.2229    0.3730      0.8082   -0.1443 -0.0452
     M1_int8 -0.0112    0.9398 -0.1381 0.7124     0.5122 -0.3489    0.7459      0.8914    0.9635  0.4987
  M1_float16  0.0600   -0.5088 -0.0030 0

## 6. Faithfulness — SHAP agreement (nsamples=1500, bootstrap CI)

In [9]:
N_PER_CLASS=60; BG_K=50; NSAMPLES=1500; B_BOOT=1000
rng=np.random.RandomState(0); sel=[]
for ci in range(N_CLASSES):
    pool=np.where(yte==ci)[0]; sel.append(rng.choice(pool,min(N_PER_CLASS,len(pool)),replace=False))
sel=np.concatenate(sel); Xev=Xte[sel]; yev=yte[sel]
bg=shap.kmeans(Xtr[rng.choice(len(Xtr),min(5000,len(Xtr)),replace=False)],BG_K)
def mkpred(path,kind):
    if kind=='keras':
        m=tf.keras.models.load_model(path); return lambda Z:m.predict(np.asarray(Z,'float32'),batch_size=4096,verbose=0)
    it=tf.lite.Interpreter(model_path=path); ii=it.get_input_details()[0]['index']; oo=it.get_output_details()[0]['index']
    def f(Z):
        Z=np.asarray(Z,'float32'); it.resize_tensor_input(ii,Z.shape); it.allocate_tensors()
        it.set_tensor(ii,Z.astype(it.get_input_details()[0]['dtype'])); it.invoke()
        Pp=it.get_tensor(oo).astype('float32'); Pp=np.clip(Pp,0,None); s=Pp.sum(1,keepdims=True); s[s==0]=1; return Pp/s
    return f
MM={'M0_s0':(anchor_path,'keras'),'M0_s1':(f"{P['models']}/M0_seed1.keras",'keras'),
    'M0_s2':(f"{P['models']}/M0_seed2.keras",'keras'),'M2_prune50':(M2[0.50],'keras'),
    'M2_prune80':(M2[0.80],'keras'),'M3_distilled':(M3_path,'keras'),
    'M1_int8':(M1_int8,'tflite'),'M1_float16':(M1_fp16,'tflite')}
def tcs(name,path,kind,es=0):
    c=f"{P['shap']}/{name}_es{es}.npy"
    if os.path.exists(c): return np.load(c)
    np.random.seed(es); sv=shap.KernelExplainer(mkpred(path,kind),bg).shap_values(Xev,nsamples=NSAMPLES,silent=True)
    if isinstance(sv,list): sv=np.stack(sv,-1)
    sv=np.asarray(sv)
    if sv.shape[-1]!=N_CLASSES and sv.shape[1]==N_CLASSES: sv=np.transpose(sv,(0,2,1))
    A=np.stack([sv[i,:,min(yev[i],sv.shape[2]-1)] for i in range(len(yev))]).astype('float32')
    np.save(c,A); print('shap',name); return A
t0=time.time()
A0=tcs('M0_s0',*MM['M0_s0'],es=0); A0b=tcs('M0_s0',*MM['M0_s0'],es=7)
A1=tcs('M0_s1',*MM['M0_s1']); A2=tcs('M0_s2',*MM['M0_s2'])
AV={n:tcs(n,*MM[n]) for n in ['M2_prune50','M2_prune80','M3_distilled','M1_int8','M1_float16']}
print(f'SHAP {(time.time()-t0)/60:.1f} min')

shap M0_s0
shap M0_s0
shap M0_s1
shap M0_s2
shap M2_prune50
shap M2_prune80
shap M3_distilled


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


shap M1_int8
shap M1_float16
SHAP 25.6 min


In [ ]:
def pss(Ar,Ao):
    o=np.full(len(Ar),np.nan)
    for i in range(len(Ar)):
        a,b=Ar[i],Ao[i]
        if np.std(a)>1e-12 and np.std(b)>1e-12: o[i]=spearmanr(a,b).correlation
    return o
def bci(v,B=B_BOOT):
    v=v[~np.isnan(v)]; r=np.random.RandomState(0); m=[v[r.randint(0,len(v),len(v))].mean() for _ in range(B)]
    return float(v.mean()),float(np.percentile(m,2.5)),float(np.percentile(m,97.5))
fm,fl,fh=bci(pss(A0,A0b)); nm,nl,nh=bci(np.concatenate([pss(A0,A1),pss(A0,A2)]))
print(f'FLOOR {fm:.3f}[{fl:.3f},{fh:.3f}] | NULL {nm:.3f}[{nl:.3f},{nh:.3f}]')
rows=[]
for n,A in AV.items():
    m,l,h=bci(pss(A0,A)); rows.append({'model':n,'spearman':round(m,3),'CI_low':round(l,3),'CI_high':round(h,3),'below_null':'YES' if h<nl else 'no'})
agt=pd.DataFrame(rows); print(agt.to_string(index=False)); agt.to_csv(f"{P['results']}/faithfulness_agreement.csv",index=False)
# per-class
def pc(Ar,Ao):
    d={}
    for ci,c in enumerate(CLASSES):
        m,l,h=bci(pss(Ar[yev==ci],Ao[yev==ci])); d[c]=f"{m:.2f}[{l:.2f},{h:.2f}]"
    return d
pcd=pd.DataFrame([{'model':n,**pc(A0,A)} for n,A in AV.items()]); print(); print(pcd.to_string(index=False))
pcd.to_csv(f"{P['results']}/faithfulness_perclass.csv",index=False)

FLOOR 0.764[0.755,0.772] | NULL 0.610[0.597,0.622]


## 7. Cross-dataset transfer summary + commit

In [ ]:
print('=== TRANSFER CHECK ===')
print('On TON_IoT, does compression collapse trust on the rare/hard classes (esp. MITM) as on CICIoT2023?')
print('- Calibration per-class gap (sec 5): watch MITM + smallest classes under prune80')
print('- Faithfulness per-class (sec 6): watch which classes drop below null band under prune80')
print('If the SAME pattern (rare classes worst) appears -> finding is a PROPERTY, not an artefact.')
print('\nresults saved:', os.listdir(P['results']))
# !git -C "{ROOT}" add ton_iot/results ton_iot/logs notebooks/
# !git -C "{ROOT}" commit -q -m "Phase D: TON_IoT cross-dataset pipeline (M0-M3, calibration, faithfulness)"
# !git -C "{ROOT}" push origin main

## Read the output
- **Anchor macro-F1 / per-class:** sanity. MITM (rare, ~1043) likely weakest — that's fine and useful.
- **Calibration per-class:** does prune80 inflate the gap most on MITM / rare classes? (transfer of the calibration collapse)
- **Faithfulness below_null + per-class:** does prune80 drop below the seed null band, worst on rare classes? (transfer of the faithfulness drift)
- **The win:** if rare classes suffer most in BOTH on TON_IoT *and* CICIoT2023, the convergence is a property of compressed IoT IDS — the top-3 multiplier.

**Next: Phase E** — deployment-like split robustness + cross-dataset figure; then Phase F figures.